# DOCX to Markdown Converter for Google Colab
This notebook converts DOCX files to Markdown format with high accuracy using multiple methods.

## Step 0: Configuration


In [ ]:
# ===============================================================
# CONFIG - EDIT ONLY THIS CELL, THEN RUNTIME > RUN ALL
# ===============================================================
from pathlib import Path

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    files = None
    IN_COLAB = False

# Choose input mode:
# - "upload": choose DOCX file(s) directly; outputs auto-download in Colab.
# - "drive": read DOCX file(s) from DRIVE_INPUT_DIR; outputs are saved to DRIVE_OUTPUT_DIR.
INPUT_MODE = "upload"
AUTO_DOWNLOAD = True

# Used for upload mode. In Colab this should normally stay /content.
DEFAULT_OUTPUT_DIR = Path("/content" if IN_COLAB else "./converted_markdown")

# Google Drive folder mode. Change these to your real Drive paths, then use INPUT_MODE="drive".
DRIVE_INPUT_DIR = Path("/content/drive/MyDrive/input_docx")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/output_markdown")
RECURSIVE_DRIVE_SEARCH = False
PRESERVE_DRIVE_SUBFOLDERS = True

# Conversion method:
# - "mammoth": best formatting/style preservation via HTML.
# - "python-docx": best for simple documents with basic structure.
# - "pandoc": best for complex documents with advanced formatting.
CONVERSION_METHOD = "mammoth"

SUPPORTED_INPUT_SUFFIXES = {".docx"}


## Step 1: Upload DOCX Files (Optional)

In [ ]:
from pathlib import Path
import os


def iter_input_files(input_dir: Path, suffixes: set[str], recursive: bool = True) -> list[Path]:
    pattern = "**/*" if recursive else "*"
    return [
        path for path in sorted(input_dir.glob(pattern))
        if path.is_file() and path.suffix.lower() in suffixes
    ]


def output_path_for(input_file: Path, output_root: Path, input_root: Path | None = None) -> Path:
    if INPUT_MODE == "drive" and PRESERVE_DRIVE_SUBFOLDERS and input_root is not None:
        try:
            relative_parent = input_file.parent.relative_to(input_root)
            target_dir = output_root / relative_parent
        except ValueError:
            target_dir = output_root
    else:
        target_dir = output_root
    target_dir.mkdir(parents=True, exist_ok=True)
    return target_dir / f"{input_file.stem}.md"


def validate_unique_output_paths(input_paths: list[Path], output_root: Path, input_root: Path | None = None) -> None:
    seen = {}
    for input_path in input_paths:
        output_path = output_path_for(input_path, output_root, input_root)
        key = str(output_path.resolve() if output_path.exists() else output_path.absolute())
        if key in seen:
            raise ValueError(
                f"Duplicate output path would be created for {seen[key]} and {input_path}: {output_path}. "
                "Set PRESERVE_DRIVE_SUBFOLDERS=True or rename one input file."
            )
        seen[key] = input_path


input_files = []
output_dir = DEFAULT_OUTPUT_DIR
input_root = None

if INPUT_MODE == "upload":
    if not IN_COLAB:
        raise RuntimeError("Direct file upload is available in Google Colab only. Use INPUT_MODE='drive' in Colab, or run this notebook in Colab.")

    output_dir.mkdir(parents=True, exist_ok=True)
    print("Choose DOCX file(s) from your computer:")
    uploaded = files.upload()

    for filename in uploaded.keys():
        file_path = Path("/content") / filename
        if file_path.suffix.lower() not in SUPPORTED_INPUT_SUFFIXES:
            raise ValueError(f"Uploaded file is not a DOCX: {filename}")
        input_files.append(file_path)

    if not input_files:
        raise ValueError("No DOCX file was uploaded.")

    print()
    print(f"Selected {len(input_files)} DOCX file(s):")
    for file_path in input_files:
        print(f"- {file_path.name} ({file_path.stat().st_size / 1024 / 1024:.2f} MB)")
elif INPUT_MODE == "drive":
    print("INPUT_MODE='drive': direct upload skipped; Drive folder will be mounted later.")
else:
    raise ValueError("INPUT_MODE must be either 'upload' or 'drive'.")


## Step 2: Install Required Libraries

In [ ]:
%pip install -q python-docx mammoth pandoc markdownify

import platform
import shutil
import subprocess

if shutil.which("pandoc") is None:
    if platform.system() == "Linux" and shutil.which("apt-get") is not None:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "pandoc"], check=True)
    else:
        print("Pandoc is not installed. Install it locally, for example: brew install pandoc")
else:
    print("Pandoc is already installed.")


## Step 3: Import Libraries

In [ ]:
import os
from pathlib import Path
from typing import Optional
import json
import docx
import mammoth
import subprocess
from markdownify import markdownify as md

if "input_files" not in globals():
    input_files = []
if "output_dir" not in globals():
    output_dir = DEFAULT_OUTPUT_DIR
if "input_root" not in globals():
    input_root = None
if "converted_files" not in globals():
    converted_files = []


## Step 4: Mount Google Drive Folder Mode (Optional)

In [ ]:
if INPUT_MODE == "drive":
    if not IN_COLAB:
        raise RuntimeError("Google Drive folder mode is available in Google Colab only.")

    from google.colab import drive

    drive.mount("/content/drive")

    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    input_root = DRIVE_INPUT_DIR
    input_files = iter_input_files(DRIVE_INPUT_DIR, SUPPORTED_INPUT_SUFFIXES, RECURSIVE_DRIVE_SEARCH)
    output_dir = DRIVE_OUTPUT_DIR

    if not input_files:
        raise FileNotFoundError(f"No DOCX files found in: {DRIVE_INPUT_DIR}")

    print("Google Drive mounted successfully.")
    print(f"Input folder: {DRIVE_INPUT_DIR}")
    print(f"Output folder: {DRIVE_OUTPUT_DIR}")
    print(f"Found {len(input_files)} DOCX file(s).")
else:
    print("INPUT_MODE='upload': Google Drive folder mode skipped.")


## Step 5: Define Conversion Functions

In [ ]:
def convert_docx_to_md_python_docx(docx_path: str, output_path: Optional[str] = None) -> str:
    """
    Convert DOCX to Markdown using python-docx (best for simple documents).

    Args:
        docx_path: Path to the DOCX file
        output_path: Optional path to save the markdown file

    Returns:
        Markdown content as string
    """
    try:
        doc = docx.Document(docx_path)
        md_content = []

        for para in doc.paragraphs:
            text = para.text.strip()
            if not text:
                continue

            # Handle headings
            if para.style.name.startswith('Heading'):
                level = para.style.name.replace('Heading ', '')
                if level.isdigit():
                    md_content.append('#' * int(level) + ' ' + text + '\n\n')
                else:
                    md_content.append(text + '\n\n')
            else:
                md_content.append(text + '\n\n')

        # Handle tables
        for table in doc.tables:
            for i, row in enumerate(table.rows):
                cells = [cell.text.strip() for cell in row.cells]
                md_content.append('| ' + ' | '.join(cells) + ' |\n')
                if i == 0:
                    md_content.append('|' + '---|' * len(cells) + '\n')
            md_content.append('\n')

        md_text = ''.join(md_content)

        if output_path:
            with open(output_path, 'w', encoding='utf-8') as f:
                f.write(md_text)
            print(f"✓ Saved to {output_path}")

        return md_text
    except Exception as e:
        print(f"✗ Error with python-docx: {str(e)}")
        return ""


def convert_docx_to_md_mammoth(docx_path: str, output_path: Optional[str] = None) -> str:
    """
    Convert DOCX to Markdown using mammoth (best for preserving formatting).

    Args:
        docx_path: Path to the DOCX file
        output_path: Optional path to save the markdown file

    Returns:
        Markdown content as string
    """
    try:
        with open(docx_path, 'rb') as docx_file:
            result = mammoth.convert_to_html(docx_file)
            html_content = result.value

            # Convert HTML to Markdown
            md_text = md(html_content)

        if output_path:
            with open(output_path, 'w', encoding='utf-8') as f:
                f.write(md_text)
            print(f"✓ Saved to {output_path}")

        return md_text
    except Exception as e:
        print(f"✗ Error with mammoth: {str(e)}")
        return ""


def convert_docx_to_md_pandoc(docx_path: str, output_path: Optional[str] = None) -> str:
    """
    Convert DOCX to Markdown using pandoc (best for complex documents).

    Args:
        docx_path: Path to the DOCX file
        output_path: Optional path to save the markdown file

    Returns:
        Markdown content as string
    """
    try:
        if not output_path:
            output_path = docx_path.replace('.docx', '.md')

        # Run pandoc command
        result = subprocess.run(
            ['pandoc', docx_path, '-t', 'markdown', '-o', output_path],
            capture_output=True,
            text=True
        )

        if result.returncode != 0:
            print(f"✗ Pandoc error: {result.stderr}")
            return ""

        with open(output_path, 'r', encoding='utf-8') as f:
            md_text = f.read()

        print(f"✓ Saved to {output_path}")
        return md_text
    except Exception as e:
        print(f"✗ Error with pandoc: {str(e)}")
        return ""


def convert_docx_to_md_hybrid(docx_path: str, output_path: Optional[str] = None, method: str = "mammoth") -> str:
    """
    Convert DOCX to Markdown using the best available method.

    Args:
        docx_path: Path to the DOCX file
        output_path: Optional path to save the markdown file
        method: "mammoth" (default), "python-docx", or "pandoc"

    Returns:
        Markdown content as string
    """
    if not os.path.exists(docx_path):
        print(f"✗ File not found: {docx_path}")
        return ""

    print(f"Converting {os.path.basename(docx_path)} using {method}...")

    if method == "mammoth":
        return convert_docx_to_md_mammoth(docx_path, output_path)
    elif method == "python-docx":
        return convert_docx_to_md_python_docx(docx_path, output_path)
    elif method == "pandoc":
        return convert_docx_to_md_pandoc(docx_path, output_path)
    else:
        print(f"✗ Unknown method: {method}")
        return ""


## Step 6: Convert with Configured Method

In [ ]:
if not input_files:
    raise ValueError("No input DOCX files. Check INPUT_MODE and the configured input source in Step 0.")

import uuid

validate_unique_output_paths(input_files, output_dir, input_root)
converted_files = []

for docx_file in input_files:
    output_path = output_path_for(docx_file, output_dir, input_root)
    temp_output_path = output_path.with_name(f".{output_path.stem}.{uuid.uuid4().hex}.tmp{output_path.suffix}")

    md_content = convert_docx_to_md_hybrid(
        str(docx_file),
        str(temp_output_path),
        method=CONVERSION_METHOD,
    )

    if not md_content.strip():
        if temp_output_path.exists():
            temp_output_path.unlink()
        raise RuntimeError(f"Conversion produced empty markdown for {docx_file.name}.")
    if not temp_output_path.exists() or temp_output_path.stat().st_size == 0:
        raise RuntimeError(f"Conversion failed or produced an empty file for {docx_file.name}.")

    temp_output_path.replace(output_path)
    converted_files.append(output_path)
    print(f"✓ Converted: {docx_file.name} -> {output_path} ({len(md_content)} characters)")

print()
print(f"✓ Conversion completed for {len(converted_files)} file(s).")
print(f"Output folder: {output_dir}")


## Step 7: Download Converted File

In [ ]:
if not converted_files:
    raise FileNotFoundError("No converted files found. Run the conversion cell first.")

if INPUT_MODE == "drive":
    print(f"Output files are saved in Google Drive: {output_dir}")
elif AUTO_DOWNLOAD:
    for output_path in converted_files:
        if not Path(output_path).exists() or Path(output_path).stat().st_size == 0:
            raise FileNotFoundError(f"Output file is missing or empty: {output_path}")
        files.download(str(output_path))
    print(f"✓ Downloaded {len(converted_files)} file(s).")
else:
    print(f"AUTO_DOWNLOAD=False. Output files are ready at: {output_dir}")
